# Food Map — Import Pipeline

**Steps:**
1. Fetch OpenAlex counts for all ~50 foods (determines node sizes in the map)
2. Browse the available foods and pick which ones to import
3. Run the import (fetches papers from OpenAlex into SQLite)
4. Build the frontend JSON files (`food_universe.json` + per-food `nodes/edges.json`)
5. Inspect what was imported

Each node in the Food Map represents a specific food (broccoli, salmon, oats, etc.).
Papers are those that mention the food in the context of infant/child nutrition.

In [1]:
import sys, os, glob, sqlite3

BACKEND_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR    = os.path.abspath(os.path.join(BACKEND_DIR, "..", "data"))
OUT_DIR     = os.path.abspath(os.path.join(BACKEND_DIR, "..", "frontend", "public"))

sys.path.insert(0, BACKEND_DIR)
from import_foods import PREDEFINED_FOODS, import_food, fetch_all_food_counts
from build_food_data import build_food_universe

print(f"Backend : {BACKEND_DIR}")
print(f"Data    : {DATA_DIR}")
print(f"Output  : {OUT_DIR}")
print(f"Foods available: {len(PREDEFINED_FOODS)}")

Backend : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\backend
Data    : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data
Output  : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\frontend\public
Foods available: 45


## Step 1 — Fetch OpenAlex paper counts

Makes one lightweight API call per food (no paper download) and saves counts to
`data/food_counts.json`. These totals drive node size in the Food Map.

Run this once upfront; re-run anytime to refresh the counts.

In [2]:
counts = fetch_all_food_counts(out_path=os.path.join(DATA_DIR, "food_counts.json"))
print(f"\nFetched counts for {len(counts)} foods.")
top10 = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 by total OpenAlex papers:")
for k, v in top10:
    cfg = PREDEFINED_FOODS.get(k, {})
    print(f"  {cfg.get('name', k):25s}  {v:>8,d}")

  [  1/45] broccoli                                  18
  [  2/45] carrot                                    23
  [  3/45] sweet_potato                              35
  [  4/45] spinach                                    1
  [  5/45] pea                                       68
  [  6/45] avocado                                   13
  [  7/45] tomato                                    37
  [  8/45] zucchini                                   2
  [  9/45] cauliflower                               12
  [ 10/45] apple                                     14
  [ 11/45] banana                                    15
  [ 12/45] mango                                     67
  [ 13/45] strawberry                                13
  [ 14/45] blueberry                                 18
  [ 15/45] pear                                       5
  [ 16/45] orange                                    58
  [ 17/45] grape                                      0
  [ 18/45] watermelon                           

## Step 2 — Browse available foods

All foods grouped by category. Foods that already have a local database show their current paper count.

In [3]:
def db_paper_count(food_key):
    path = os.path.join(DATA_DIR, f'food_papers_{food_key}.db')
    if not os.path.exists(path):
        return None
    try:
        conn = sqlite3.connect(path)
        n = conn.execute('SELECT COUNT(*) FROM papers').fetchone()[0]
        conn.close()
        return n
    except Exception:
        return 0

from import_foods import FOOD_GROUP_ORDER

# Group foods by their category
by_group = {g: [] for g in FOOD_GROUP_ORDER}
for key, cfg in PREDEFINED_FOODS.items():
    by_group.setdefault(cfg['group'], []).append((key, cfg))

total_imported = 0
for group in FOOD_GROUP_ORDER:
    foods = by_group.get(group, [])
    if not foods:
        continue
    print(f'\n── {group.upper()} ──')
    for key, cfg in foods:
        count = db_paper_count(key)
        status = f'{count:4d} papers' if count is not None else '   (not imported)'
        print(f'  {key:25s}  {cfg["name"]:25s}  {status}')
        if count:
            total_imported += count

existing = len(glob.glob(os.path.join(DATA_DIR, 'food_papers_*.db')))
print(f'\n{existing} food databases on disk, {total_imported:,} papers total')


── VEGETABLES ──
  broccoli                   Broccoli                      (not imported)
  carrot                     Carrot                        (not imported)
  sweet_potato               Sweet Potato                  (not imported)
  spinach                    Spinach                       (not imported)
  pea                        Peas                          (not imported)
  avocado                    Avocado                       (not imported)
  tomato                     Tomato                        (not imported)
  zucchini                   Zucchini                      (not imported)
  cauliflower                Cauliflower                   (not imported)

── FRUITS ──
  apple                      Apple                         (not imported)
  banana                     Banana                        (not imported)
  mango                      Mango                         (not imported)
  strawberry                 Strawberry                    (not imported)
  blue

## Step 3 — Configure & import

Edit `FOODS_TO_IMPORT` to select which foods to fetch.  
Use `list(PREDEFINED_FOODS.keys())` to import all foods.

`MAX_PAPERS` controls how many papers to fetch per food from OpenAlex.  
`MIN_CITATIONS` filters out papers with fewer citations (0 = include everything).

In [4]:
# ── configure here ────────────────────────────────────────────────────────────

# Import a hand-picked selection:
# FOODS_TO_IMPORT = [
#     'broccoli',
#     'carrot',
#     'sweet_potato',
#     'apple',
#     'banana',
#     'salmon',
#     'egg',
#     'chicken',
#     'oats',
#     'lentil',
#     'yogurt',
#     'breast_milk',
#     'avocado',
#     'peanut',
# ]

# Or import everything:
FOODS_TO_IMPORT = list(PREDEFINED_FOODS.keys())

MAX_PAPERS    = 300   # papers per food
MIN_CITATIONS = 0     # set higher (e.g. 5) to skip low-impact papers
SKIP_EXISTING = True  # skip foods that already have a database

# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DATA_DIR, exist_ok=True)

to_run = []
for key in FOODS_TO_IMPORT:
    if key not in PREDEFINED_FOODS:
        print(f'  [warn] unknown food key: {key} — skipping')
        continue
    existing_count = db_paper_count(key)
    if SKIP_EXISTING and existing_count is not None:
        print(f'  [skip] {key} — already has {existing_count} papers')
        continue
    to_run.append(key)

print(f'\nWill import {len(to_run)} food(s): {", ".join(to_run)}')


Will import 45 food(s): broccoli, carrot, sweet_potato, spinach, pea, avocado, tomato, zucchini, cauliflower, apple, banana, mango, strawberry, blueberry, pear, orange, grape, watermelon, chicken, beef, salmon, sardine, egg, lentil, cows_milk, yogurt, cheese, breast_milk, oats, rice, wheat, quinoa, corn, chickpea, kidney_bean, peanut, soy, almond, olive_oil, coconut_oil, flaxseed, probiotic_food, prebiotic_food, dark_chocolate, herbs_spices


In [5]:
results = {}
for i, key in enumerate(to_run, 1):
    cfg = PREDEFINED_FOODS[key]
    print(f'\n[{i}/{len(to_run)}] {cfg["name"]} ({key})')
    print(f'  Query: "{cfg["query"]}')
    try:
        import_food(key, max_results=MAX_PAPERS, min_citations=MIN_CITATIONS)
        count = db_paper_count(key) or 0
        results[key] = count
        print(f'  → {count} papers in database')
    except Exception as e:
        print(f'  [error] {e}')
        results[key] = 0

print(f'\nDone. {sum(results.values()):,} papers imported across {len(results)} foods.')


[1/45] Broccoli (broccoli)
  Query: "broccoli infant child vegetable introduction nutrition cancer glucosinolate
[import] Food:  Broccoli
[import] Query: broccoli infant child vegetable introduction nutrition cancer glucosinolate
[import] DB:    C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data\food_papers_broccoli.db
[import] Existing papers: 0
[import] Total in OpenAlex: 18
  Fetched 18 papers total.   
[import] Done: 18 inserted, 0 skipped
  → 18 papers in database

[2/45] Carrot (carrot)
  Query: "carrot infant child vegetable beta-carotene vitamin A introduction puree
[import] Food:  Carrot
[import] Query: carrot infant child vegetable beta-carotene vitamin A introduction puree
[import] DB:    C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data\food_papers_carrot.db
[import] Existing papers: 0
[import] Total in OpenAlex: 23
  Fetched 23 papers total.   
[import] Done: 23 inserted, 0 skipped
  → 23 papers in database

[3/45] Sweet Potato (sweet_potato)
  Query: "sweet potato inf

## Step 4 — Build frontend JSON

Reads all `food_papers_*.db` files and writes:
- `frontend/public/food_universe.json` — the food map index
- `frontend/public/food_data/<food>/nodes.json`
- `frontend/public/food_data/<food>/edges.json`

Refresh the browser after this runs.

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)
build_food_universe(DATA_DIR, OUT_DIR)
print('\nFrontend JSON built. Refresh http://localhost:5173 to see the Food Map.')

[build] Found 45 food databases
[build] Processing almond...
  [ok] almond: 26 papers, 0 citations
[build] Processing apple...
  [ok] apple: 14 papers, 0 citations
[build] Processing avocado...
  [ok] avocado: 13 papers, 0 citations
[build] Processing banana...
  [ok] banana: 15 papers, 3 citations
[build] Processing beef...
  [ok] beef: 15 papers, 2 citations
[build] Processing blueberry...
  [ok] blueberry: 18 papers, 2 citations
[build] Processing breast_milk...
  [ok] breast_milk: 300 papers, 291 citations
[build] Processing broccoli...
  [ok] broccoli: 18 papers, 0 citations
[build] Processing carrot...
  [ok] carrot: 23 papers, 1 citations
[build] Processing cauliflower...
  [ok] cauliflower: 12 papers, 0 citations
[build] Processing cheese...
  [ok] cheese: 67 papers, 0 citations
[build] Processing chicken...
  [ok] chicken: 143 papers, 19 citations
[build] Processing chickpea...
  [ok] chickpea: 2 papers, 0 citations
[build] Processing coconut_oil...
  [ok] coconut_oil: 27 pape

## Step 5 — Inspect what was imported

In [7]:
dbs = sorted(glob.glob(os.path.join(DATA_DIR, 'food_papers_*.db')))
if not dbs:
    print('No food databases found. Run Step 3 first.')
else:
    total = 0
    print(f'{"Food":25s} {"Group":12s} {"Papers":>8s} {"Max Citations":>14s} {"Year range":>12s}')
    print('-' * 80)
    for db_path in dbs:
        key = os.path.basename(db_path).replace('food_papers_', '').replace('.db', '')
        cfg = PREDEFINED_FOODS.get(key, {})
        conn = sqlite3.connect(db_path)
        try:
            row = conn.execute(
                'SELECT COUNT(*) as n, MAX(cited_by_count) as max_c, '
                'MIN(year) as yr0, MAX(year) as yr1 FROM papers'
            ).fetchone()
            n, max_c, yr0, yr1 = row
            yr_range = f'{yr0}–{yr1}' if yr0 and yr1 else 'unknown'
            print(f'{cfg.get("name", key):25s} {cfg.get("group", ""):12s} {n:>8,d} {(max_c or 0):>14,d} {yr_range:>12s}')
            total += n
        except Exception as e:
            print(f'{key}: error — {e}')
        finally:
            conn.close()
    print('-' * 80)
    print(f'{"TOTAL":25s} {"":12s} {total:>8,d}')

Food                      Group          Papers  Max Citations   Year range
--------------------------------------------------------------------------------
Almond                    legumes            26            287    2011–2025
Apple                     fruits             14            377    2002–2025
Avocado                   vegetables         13             44    2014–2025
Banana                    fruits             15            248    2014–2026
Beef                      proteins           15            295    2004–2024
Blueberry                 fruits             18             44    2003–2026
Breast Milk               dairy             300          5,848    1994–2025
Broccoli                  vegetables         18            377    2003–2026
Carrot                    vegetables         23            377    2012–2025
Cauliflower               vegetables         12            377    2003–2026
Cheese                    dairy              67            101    1922–2026
Chicken